# 05 - Algorithm Comparison

**Purpose**: Compare all clustering algorithms side-by-side

**Comparisons**:
- Quality metrics (Silhouette, Calinski-Harabasz)
- Agreement between algorithms (ARI)
- Cluster size distributions
- Visual comparisons in PCA space

---

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, silhouette_score, calinski_harabasz_score
from sklearn.decomposition import PCA

from notebook_utils import setup_notebook

%matplotlib inline
print("✓ Imports loaded")

In [ ]:
# Load state
cfg, state = setup_notebook("Algorithm Comparison", "germany")

# Load all clustering results
kmeans_labels = state.load('kmeans_labels')
hierarchical_labels = state.load('hierarchical_labels')
dbscan_labels = state.load('dbscan_labels')

kmeans_metrics = state.load('kmeans_metrics', {})
hierarchical_metrics = state.load('hierarchical_metrics', {})
dbscan_metrics = state.load('dbscan_metrics', {})

# Load data
df_latest = state.load('df_latest')
feature_cols = state.load('feature_cols')
kmeans_scaler = state.load('kmeans_scaler')

X_scaled = kmeans_scaler.transform(df_latest[feature_cols].fillna(df_latest[feature_cols].median()))

print("✓ Data loaded")

## 1. Quality Metrics Comparison

In [ ]:
# Calculate metrics for all algorithms
metrics_data = []

if kmeans_labels is not None:
    metrics_data.append({
        'Algorithm': 'K-Means',
        'Silhouette': kmeans_metrics.get('silhouette', silhouette_score(X_scaled, kmeans_labels)),
        'Calinski-Harabasz': kmeans_metrics.get('calinski_harabasz', calinski_harabasz_score(X_scaled, kmeans_labels)),
        'N_Clusters': kmeans_metrics.get('k', len(set(kmeans_labels)))
    })

if hierarchical_labels is not None:
    metrics_data.append({
        'Algorithm': 'Hierarchical',
        'Silhouette': hierarchical_metrics.get('silhouette', silhouette_score(X_scaled, hierarchical_labels)),
        'Calinski-Harabasz': calinski_harabasz_score(X_scaled, hierarchical_labels),
        'N_Clusters': hierarchical_metrics.get('n_clusters', len(set(hierarchical_labels)))
    })

if dbscan_labels is not None:
    # Exclude outliers for DBSCAN metrics
    mask = dbscan_labels != -1
    if mask.sum() > 0 and len(set(dbscan_labels[mask])) > 1:
        metrics_data.append({
            'Algorithm': 'DBSCAN',
            'Silhouette': silhouette_score(X_scaled[mask], dbscan_labels[mask]),
            'Calinski-Harabasz': calinski_harabasz_score(X_scaled[mask], dbscan_labels[mask]),
            'N_Clusters': dbscan_metrics.get('n_clusters', len(set(dbscan_labels)) - 1)
        })

df_metrics = pd.DataFrame(metrics_data)

print("\n📊 Quality Metrics Comparison:")
print("=" * 80)
display(df_metrics)
print("=" * 80)

In [ ]:
# Visualize metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette scores
df_metrics.plot(x='Algorithm', y='Silhouette', kind='bar', ax=ax1, 
               color='steelblue', alpha=0.7, legend=False)
ax1.set_ylabel('Silhouette Score', fontsize=12)
ax1.set_title('Silhouette Score Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# Calinski-Harabasz scores
df_metrics.plot(x='Algorithm', y='Calinski-Harabasz', kind='bar', ax=ax2,
               color='orange', alpha=0.7, legend=False)
ax2.set_ylabel('Calinski-Harabasz Score', fontsize=12)
ax2.set_title('Calinski-Harabasz Score Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 2. Agreement Between Algorithms (ARI)

In [ ]:
# Calculate pairwise ARI
algorithms = []
labels_list = []

if kmeans_labels is not None:
    algorithms.append('K-Means')
    labels_list.append(kmeans_labels)

if hierarchical_labels is not None:
    algorithms.append('Hierarchical')
    labels_list.append(hierarchical_labels)

if dbscan_labels is not None:
    algorithms.append('DBSCAN')
    labels_list.append(dbscan_labels)

# Create ARI matrix
n_algs = len(algorithms)
ari_matrix = np.ones((n_algs, n_algs))

for i in range(n_algs):
    for j in range(i+1, n_algs):
        ari = adjusted_rand_score(labels_list[i], labels_list[j])
        ari_matrix[i, j] = ari
        ari_matrix[j, i] = ari

df_ari = pd.DataFrame(ari_matrix, index=algorithms, columns=algorithms)

print("\n📊 Adjusted Rand Index (ARI) Matrix:")
print("=" * 80)
display(df_ari)
print("\nInterpretation: 1.0 = perfect agreement, 0.0 = random agreement")
print("=" * 80)

In [ ]:
# Visualize ARI as heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_ari, annot=True, fmt='.3f', cmap='RdYlGn', 
           vmin=0, vmax=1, square=True, linewidths=1,
           cbar_kws={'label': 'ARI Score'})
plt.title('Algorithm Agreement (Adjusted Rand Index)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Visual Comparison in PCA Space

In [ ]:
# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
print(f"PCA Explained Variance: PC1={explained_var[0]:.1%}, PC2={explained_var[1]:.1%}")

In [ ]:
# Plot all algorithms side-by-side
fig, axes = plt.subplots(1, len(algorithms), figsize=(6*len(algorithms), 5))

if len(algorithms) == 1:
    axes = [axes]

for idx, (alg, labels) in enumerate(zip(algorithms, labels_list)):
    ax = axes[idx]
    
    # Handle DBSCAN outliers
    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    
    colors = plt.cm.nipy_spectral(np.linspace(0, 1, n_clusters))
    
    for k, col in zip(range(n_clusters), colors):
        class_member_mask = (labels == k)
        ax.scatter(X_pca[class_member_mask, 0], X_pca[class_member_mask, 1],
                  c=[col], alpha=0.6, s=30, edgecolors='k', linewidth=0.3,
                  label=f'Cluster {k}')
    
    # Plot outliers for DBSCAN
    if -1 in labels:
        outlier_mask = (labels == -1)
        ax.scatter(X_pca[outlier_mask, 0], X_pca[outlier_mask, 1],
                  c='gray', alpha=0.3, s=20, marker='x',
                  label='Outliers')
    
    ax.set_xlabel(f'PC1 ({explained_var[0]:.1%})', fontsize=11)
    ax.set_ylabel(f'PC2 ({explained_var[1]:.1%})', fontsize=11)
    ax.set_title(f'{alg}\n({n_clusters} clusters)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Cluster Size Distributions

In [ ]:
# Compare cluster size distributions
fig, axes = plt.subplots(1, len(algorithms), figsize=(6*len(algorithms), 4))

if len(algorithms) == 1:
    axes = [axes]

for idx, (alg, labels) in enumerate(zip(algorithms, labels_list)):
    ax = axes[idx]
    
    # Count labels (excluding outliers)
    unique, counts = np.unique(labels[labels != -1], return_counts=True)
    
    ax.bar(unique, counts, color='steelblue', alpha=0.7)
    ax.set_xlabel('Cluster ID', fontsize=11)
    ax.set_ylabel('Number of Companies', fontsize=11)
    ax.set_title(f'{alg}\nCluster Sizes', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, (cluster_id, count) in enumerate(zip(unique, counts)):
        ax.text(cluster_id, count + 5, str(count), 
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Summary & Recommendations

In [ ]:
print("\n" + "=" * 80)
print("  ALGORITHM COMPARISON SUMMARY")
print("=" * 80)

# Best algorithm by Silhouette
if len(df_metrics) > 0:
    best_silhouette = df_metrics.loc[df_metrics['Silhouette'].idxmax()]
    print(f"\n✓ Best Silhouette Score: {best_silhouette['Algorithm']} ({best_silhouette['Silhouette']:.3f})")
    
    best_calinski = df_metrics.loc[df_metrics['Calinski-Harabasz'].idxmax()]
    print(f"✓ Best Calinski-Harabasz: {best_calinski['Algorithm']} ({best_calinski['Calinski-Harabasz']:.2f})")

print("\n📊 Algorithm Characteristics:")
print("  K-Means:       Fast, spherical clusters, requires K")
print("  Hierarchical:  Hierarchical structure, no K required upfront")
print("  DBSCAN:        Density-based, finds outliers, arbitrary shapes")

print("\n💡 Recommendation:")
print("  Use K-Means for: Fast, well-separated clusters")
print("  Use Hierarchical for: Exploring cluster relationships")
print("  Use DBSCAN for: Finding outliers and irregular shapes")

print("\n📝 Next: 06_Deep_Analysis.ipynb for detailed insights")
print("=" * 80)